In [1]:
# Semantic chunking and create LLM friendly table row

In [1]:
from pdf_with_tables import (get_page_elements, check_and_fix_nested_tables,
                             change_table_format)

In [6]:
pdf_pages = get_page_elements('assets/CP-Bantuan-Pengobatan-2025.pdf')

In [7]:
pdf_pages = check_and_fix_nested_tables(pdf_pages)
pdf_pages = change_table_format(pdf_pages)
pdf_pages

[{'page': 1,
  'narrative_text': 'KPN CORP\nBANTUAN PENGOBATAN\nFEB 2025\nDEFINITION\n● BANTUAN PENGOBATAN adalah bantuan biaya kesehatan di luar BPJS Kesehatan\nyang diberikan Perusahaan kepada karyawan beserta keluarganya yang sakit dan\nmemerlukan pengobatan oleh tenaga medis atau dokter yang meliputi Bantuan\nPengobatan Rawat Jalan, Bantuan Pengobatan Rawat Inap, Bantuan Biaya Persalinan,\nBantuan Kacamata, dan Bantuan Pengobatan Kesehatan Khusus. 1\n● BANTUAN PENGOBATAN RAWAT JALAN adalah bantuan biaya kesehatan yang\ndiberikan Perusahaan kepada karyawan dan keluarganya yang sakit dan memerlukan\n2\npengobatan oleh tenaga medis atau dokter tetapi tidak memerlukan rawat inap.\n● BANTUAN RAWAT INAP adalah bantuan biaya kesehatan yang diberikan Perusahaan\nkepada karyawan dan keluarganya yang sakit serta berdasarkan rekomendasi dan\ndiagnosa dokter memerlukan perawatan inap di rumah sakit atau fasilitas medis lainnya\nyang ditentukan oleh dokter yang menangani masalah kesehatan terse

In [8]:
import re

def semantic_chunk(text, chunk_size=300, chunk_overlap=60):
    """
    Chunk text semantically: preserve headings, bullets, and numbered items,
    then merge into chunks with overlap. Falls back to sentence splitting
    if a single unit is longer than chunk_size.
    """

    # Split into atomic units
    lines = re.split(r'\n+', text)
    units = []
    buffer = []

    for line in lines:
        line = line.strip()
        if not line:
            continue

        # Heuristic: treat as a new unit if bullet/number/heading
        if re.match(r'^[●\-\*]\s+', line):          # bullet point
            if buffer:
                units.append(" ".join(buffer).strip())
                buffer = []
            units.append(line)

        elif re.match(r'^\d+[\.\)]?\s+', line):     # numbered item
            if buffer:
                units.append(" ".join(buffer).strip())
                buffer = []
            units.append(line)

        elif line.isupper() and len(line.split()) <= 6:  # short all-caps heading
            if buffer:
                units.append(" ".join(buffer).strip())
                buffer = []
            units.append(line)

        else:
            buffer.append(line)

    if buffer:
        units.append(" ".join(buffer).strip())

    # Merge into chunks with overlap
    chunks = []
    current = ""

    for unit in units:
        if len(current) + len(unit) + 1 <= chunk_size:
            current += " " + unit if current else unit
        else:
            if current:
                chunks.append(current.strip())
            # If unit itself too long, break by sentences
            if len(unit) > chunk_size:
                sentences = re.split(r'(?<=[.!?])\s+', unit)
                temp = ""
                for sent in sentences:
                    if len(temp) + len(sent) + 1 <= chunk_size:
                        temp += " " + sent if temp else sent
                    else:
                        chunks.append(temp.strip())
                        temp = sent
                if temp:
                    chunks.append(temp.strip())
                current = ""
            else:
                current = unit

    if current:
        chunks.append(current.strip())

    # Add overlap
    if chunk_overlap > 0 and len(chunks) > 1:
        overlapped = []
        for i, chunk in enumerate(chunks):
            if i == 0:
                overlapped.append(chunk)
            else:
                prev = overlapped[-1]
                tail = prev[-chunk_overlap:]
                overlapped.append(tail + " " + chunk)
        chunks = overlapped

    return chunks


In [11]:
chunks = semantic_chunk(pdf_pages[3]['narrative_text'])
chunks

['9. Pap smears, apabila dilakukan atas rekomendasi dokter. 49 10. 50 Fisioterapi. 11. Tindakan-tindakan rawat jalan lain yang diakui secara medis dan dibuktikan dengan surat keterangan dari dokter. 51 52 BANTUAN PENGOBATAN TINGKAT LANJUTAN 1. Pemeriksaan dan pengobatan oleh dokter spesialis.',
 'ANJUTAN 1. Pemeriksaan dan pengobatan oleh dokter spesialis. 2. Pemeriksaan atau tindakan diagnostik lanjutan seperti EEG, SCG, EMG, CT Scan, endoscopy, USG, patologi anatomi, radiologi, atau tindakan diagnostik lainnya atas rekomendasi dokter yang merawat dan berkaitan dengan penyakit yang diderita. 54545454',
 'erawat dan berkaitan dengan penyakit yang diderita. 54545454 3. Pemberian obat-obatan yang berpedoman pada Daftar Obat Esensial National Plus 55 (DOEN Plus) atau generik sesuai dengan resep dokter. 56 BANTUAN PENGOBATAN RAWAT INAP ● Diberikan Kepada: 57 1. Karyawan Tetap dan Karyawan Kontrak, sejak hari pertama bekerja di Perusahaan. 58',
 'ryawan Kontrak, sejak hari pertama bekerja d

In [21]:
from typing import Any, Dict
import re
import unicodedata

In [14]:
def canonicalize_text(s: str, lower: bool = False) -> str:
    """Normalize text for stable checksum: unicode, collapse whitespace, optional lowercase."""
    if s is None:
        s = ""
    s = unicodedata.normalize("NFC", str(s))
    s = s.strip()
    s = re.sub(r"\s+", " ", s).strip()
    # normalize smart quotes
    s = s.replace("“", '"').replace("”", '"').replace("’", "'").replace("`", "'")
    # normalize common dash placeholders
    if s in ["-", "—", "–", "n/a", "N/A", "NA", "null", "None"]:
        return ""
    return s

In [15]:
def row_dict_to_text(row: Dict[str, Any]) -> str:
    """
    Convert a row dict into a stable, LLM- and vector-db-friendly text string.
    - Keeps the same signature and return type as your original function.
    - Produces natural-language sentences (one per non-empty cell) joined by newlines.
    - Deterministic ordering: keys sorted case-insensitively.
    - Canonicalizes whitespace and normalizes common "empty" tokens.
    - If a common primary key like 'Gol.'/'Gol'/'ID' exists, sentences are prefixed with it
      (e.g. "For Gol. 9, the value for 'Header' is ...").
    """
   

    if not isinstance(row, dict):
        return canonicalize_text(row)

    # deterministic key order
    keys = sorted(row.keys(), key=lambda k: str(k).lower())

    # detect a primary-key-like column if present
    pk_candidates = [k for k in keys if str(k).lower() in {"gol.", "gol", "id", "no", "nomor", "kode"}]
    primary_key = pk_candidates[0] if pk_candidates else None
    primary_val = canonicalize_text(row.get(primary_key)) if primary_key else ""

    sentences = []
    for k in keys:
        # canonicalize header and value
        header = canonicalize_text(k)
        value = canonicalize_text(row.get(k))
        # skip empties
        if value == "":
            continue
        # keep header readable but bounded
        if len(header) > 120:
            header = header[:80].rstrip() + " … " + header[-30:].lstrip()
        # create LLM-friendly sentence; prefer "For <PK>" when available and this isn't the PK cell itself
        if primary_key and k != primary_key and primary_val:
            sentence = f"For {primary_key} {primary_val}, the value for '{header}' is {value}."
        elif primary_key and k == primary_key:
            # describe the primary key itself
            sentence = f"{primary_key}: {value}."
        else:
            sentence = f"The value for '{header}' is {value}."
        sentences.append(sentence)

    # fallback: if everything was empty (or only primary present but skipped), return canonicalized row repr
    if not sentences:
        # produce compact "k: v | k2: v2" style as last resort (deterministic)
        parts = []
        for k in keys:
            v = canonicalize_text(row.get(k))
            parts.append(f"{k}: {v}" if v else f"{k}:")
        return " | ".join(parts)

    # join sentences with newline to keep single-string output while preserving chunk granularity
    return "\n".join(sentences)


In [16]:
pdf_pages = get_page_elements('assets/CP_Perjalanan Dinas.pdf')
pdf_pages = check_and_fix_nested_tables(pdf_pages)
pdf_pages = change_table_format(pdf_pages)
pdf_pages

[{'page': 1, 'narrative_text': 'PERJALANAN DINAS\nAUG 2025', 'tables': []},
 {'page': 2,
  'narrative_text': 'DEFINITION\nPERJALANAN DINAS adalah perjalanan yang dilakukan oleh Karyawan Perusahaan menuju lokasi di luar Lokasi Kerja Karyawan dan/atau kembali\nkeLokasiKerjaKaryawandalamrangkapelaksanaantugasataukepentinganPerusahaan.\nPERJALANAN DINAS JARAK DEKAT, disingkat sebagai PDJD, adalah Perjalanan Dinas dalam rangka pelaksanaan tugas atau kepentingan\nPerusahaan ke suatu wilayah dengan jarak antara Lokasi Kerja Karyawan dengan tujuan dinas <125 km (kurang dari seratus dua puluh lima\nkilometer)dalam1(satu)kaliperjalanan.\nPERJALANANDINASJARAKJAUH,disingkatsebagaiPDJJ,adalahPerjalananDinas dalamrangkapelaksanaantugasataukepentinganPerusahaan\nke suatu wilayah di dalam negeri maupun luar negeri negeri dengan jarak antara Lokasi Kerja Karyawan dengan tujuan dinas ≥125 km (lebih dari\natausamadenganseratusduapuluhlimakilometer)dalam1(satu)kaliperjalanandandapatmemilikiunsurmenginap.\

In [19]:
table_test = pdf_pages[-1]['tables'][0]['table_dct']
table_test

[{'Gol.': '10',
  'Dalam Negeri (dalam ribuan Rupiah) - Sumatera': 'Sesuai tagihan, Hotel Bintang 5 (lima)',
  'Dalam Negeri (dalam ribuan Rupiah) - Jawa & Bali': 'Sesuai tagihan, Hotel Bintang 5 (lima)',
  'Dalam Negeri (dalam ribuan Rupiah) - Kalimantan (selain Kaltim)': 'Sesuai tagihan, Hotel Bintang 5 (lima)',
  'Dalam Negeri (dalam ribuan Rupiah) - Kaltim & Sulawesi': 'Sesuai tagihan, Hotel Bintang 5 (lima)',
  'Dalam Negeri (dalam ribuan Rupiah) - Papua': 'Sesuai tagihan, Hotel Bintang 5 (lima)',
  'Luar Negeri (dalam USD) - Asia1), Afrika (dalam USD)': 'Sesuai tagihan, Hotel Bintang 5 (lima)',
  'Luar Negeri (dalam USD) - Non Asia& Afrika, Singapore, Hongkong dan Jepang (dalam USD)': 'Sesuai tagihan, Hotel Bintang 5 (lima)'},
 {'Gol.': '9',
  'Dalam Negeri (dalam ribuan Rupiah) - Sumatera': '1.200',
  'Dalam Negeri (dalam ribuan Rupiah) - Jawa & Bali': '1.500',
  'Dalam Negeri (dalam ribuan Rupiah) - Kalimantan (selain Kaltim)': '1.200',
  'Dalam Negeri (dalam ribuan Rupiah) - K

In [24]:
text = row_dict_to_text(table_test[1])
print(text)

For Gol. 9, the value for 'Dalam Negeri (dalam ribuan Rupiah) - Jawa & Bali' is 1.500.
For Gol. 9, the value for 'Dalam Negeri (dalam ribuan Rupiah) - Kalimantan (selain Kaltim)' is 1.200.
For Gol. 9, the value for 'Dalam Negeri (dalam ribuan Rupiah) - Kaltim & Sulawesi' is 1.200.
For Gol. 9, the value for 'Dalam Negeri (dalam ribuan Rupiah) - Papua' is 1.500.
For Gol. 9, the value for 'Dalam Negeri (dalam ribuan Rupiah) - Sumatera' is 1.200.
Gol.: 9.
For Gol. 9, the value for 'Luar Negeri (dalam USD) - Asia1), Afrika (dalam USD)' is 300.
For Gol. 9, the value for 'Luar Negeri (dalam USD) - Non Asia& Afrika, Singapore, Hongkong dan Jepang (dalam USD)' is 400.
